In [91]:
import pandas as pd

data_full = pd.read_csv("pitch_data_full.csv")

In [92]:
print(data_full.columns)

Index(['pitch_type', 'game_date', 'release_speed', 'release_pos_x',
       'release_pos_z', 'player_name', 'batter', 'pitcher', 'events',
       'description',
       ...
       'batter_days_until_next_game', 'api_break_z_with_gravity',
       'api_break_x_arm', 'api_break_x_batter_in', 'arm_angle', 'attack_angle',
       'attack_direction', 'swing_path_tilt',
       'intercept_ball_minus_batter_pos_x_inches',
       'intercept_ball_minus_batter_pos_y_inches'],
      dtype='object', length=119)


In [93]:
import numpy as np

woba_scale = {2026: 1.238, 2025: 1.232, 2024: 1.242,2023: 1.204, 2022: 1.259, 2021: 1.209}
data_full['woba_scale'] = data_full["game_year"].map(woba_scale)

def add_pitch_score(df):
    pa = ["game_pk", "at_bat_number"]
    df = (df[df["game_type"] == "R"]
          .sort_values(pa + ["pitch_number"])
          .copy())
    df["count"] = df["balls"].astype(str) + "-" + df["strikes"].astype(str)

    last = df.groupby(pa).tail(1)
    last = last[last["woba_denom"] == 1]
    pa_value = np.where(
        last["type"].eq("X") & last["estimated_woba_using_speedangle"].notna(),
        last["estimated_woba_using_speedangle"],   # ball in play: xwOBA
        last["woba_value"],                        # K, BB, HBP: actual value
    )
    last = last.assign(pa_value=pa_value)[pa + ["pa_value"]]
    df = df.merge(last, on=pa, how="left")

    seen = df.dropna(subset=["pa_value"]).drop_duplicates(pa + ["count"])
    count_woba = seen.groupby("count")["pa_value"].mean()

    df["woba_before"] = df["count"].map(count_woba)
    next_count = df.groupby(pa)["count"].shift(-1)
    df["woba_after"] = next_count.map(count_woba)
    is_last = next_count.isna()
    df.loc[is_last, "woba_after"] = df.loc[is_last, "pa_value"]

    df["pitch_score"] = (df["woba_before"] - df["woba_after"]) / df["woba_scale"]
    return df, count_woba

data_full, count_woba = add_pitch_score(data_full)
print(count_woba.round(3))

count
0-0    0.317
0-1    0.268
0-2    0.200
1-0    0.359
1-1    0.300
1-2    0.223
2-0    0.428
2-1    0.359
2-2    0.269
3-0    0.551
3-1    0.480
3-2    0.376
Name: pa_value, dtype: float64


In [94]:
keep_cols = [
    "pitch_score",
    "delta_pitcher_run_exp",
    "release_speed", "release_spin_rate", "spin_axis",
    "pfx_x", "pfx_z",
    "plate_x", "plate_z", "sz_top", "sz_bot",
    "balls", "strikes", "stand", "p_throws",
    "pitcher", "game_year", "pitch_type","game_type"
]

data = data_full[keep_cols].copy()
data = data.dropna().copy()

data.columns

Index(['pitch_score', 'delta_pitcher_run_exp', 'release_speed',
       'release_spin_rate', 'spin_axis', 'pfx_x', 'pfx_z', 'plate_x',
       'plate_z', 'sz_top', 'sz_bot', 'balls', 'strikes', 'stand', 'p_throws',
       'pitcher', 'game_year', 'pitch_type', 'game_type'],
      dtype='object')

In [95]:
data = data[data["game_type"] == "R"]
data["plate_z_normalized"] = (data["plate_z"] - data["sz_bot"]) / (data["sz_top"] - data["sz_bot"])
n = data.groupby(["pitcher", "game_year"])["release_speed"].transform("size")
data = data[n >= 200].copy()

grp = data.groupby(["pitcher", "game_year"])
data["velo_pctile"] = grp["release_speed"].rank(pct=True)
data["spin_pctile"] = grp["release_spin_rate"].rank(pct=True)

In [97]:
data.to_csv("pitch_data_processed.csv")

In [98]:
data.columns

Index(['pitch_score', 'delta_pitcher_run_exp', 'release_speed',
       'release_spin_rate', 'spin_axis', 'pfx_x', 'pfx_z', 'plate_x',
       'plate_z', 'sz_top', 'sz_bot', 'balls', 'strikes', 'stand', 'p_throws',
       'pitcher', 'game_year', 'pitch_type', 'game_type', 'plate_z_normalized',
       'velo_pctile', 'spin_pctile'],
      dtype='object')